# symmetry.py — Full Walkthrough

This notebook explains every part of `symmetry.py` from the Intertwined Lattices project.
The file is about representing, analysing, and manipulating **3-D lattice structures** (like metamaterial unit cells).

It does five main things:
1. **Read** a lattice from a `.dat` text file
2. **Classify** every node as a *port*, *counterport*, or *junction*
3. **Find symmetries** of the point cloud using graph isomorphism
4. **Tessellate** (tile) the unit cell to fill a larger region of space
5. Wrap everything into a **`Graph` class** that keeps all data organised and offers visualisation methods

Work through each section below to understand what each function receives and what it returns.

## 1 — Imports and Dependencies

The file relies on the following libraries:

| Library | Purpose |
|---|---|
| `numpy` | Numerical arrays, linear algebra, distance calculations |
| `matplotlib` | 2-D and 3-D plotting of lattices and graphs |
| `mpl_toolkits.mplot3d.art3d.Line3DCollection` | Efficient drawing of many 3-D line segments at once |
| `networkx` | Graph data structure and the **isomorphism engine** used to find symmetries |
| `networkx.algorithms.isomorphism.GraphMatcher` | Finds all ways a graph can be mapped onto itself (automorphisms) |
| `itertools.product` | Generates every combination of tessellation multiplicity indices |
| `collections.defaultdict` | A dictionary that auto-creates a default value for missing keys |
| `export` (local) | Project-local helpers for writing output files (e.g. `mesh2dat`) |

In [ ]:
import numpy as np
from mpl_toolkits.mplot3d.art3d import Line3DCollection
import sys
import matplotlib.pyplot as plt
import networkx as nx
from networkx.algorithms.isomorphism import GraphMatcher
from itertools import product
from collections import defaultdict

# Tell matplotlib to show plots inline in the notebook
%matplotlib inline

print("All dependencies imported successfully.")

## 2 — Reading Lattice Files: `read_lattice_file()`

### What the `.dat` file looks like

The input file has two sections separated by the keyword `connectivity`:

```
x,y,z        ← one 3-D point per line
x,y,z
...
connectivity
start,end    ← indices into the points list (0-based)
start,end
...
```

An optional third section `counterports` can also appear to explicitly declare which nodes are counterports
(nodes that sit on the *opposite* face of the unit cell from their paired port).

---

### The three node types

| Type | Meaning | How it is detected |
|---|---|---|
| **port** | A boundary node — it sits on the surface of the unit cell and is where the lattice connects to its neighbours when tiled | Connected to exactly **1** other node (degree = 1) |
| **counterport** | The matching boundary node on the *opposite* face — it is the periodic partner of a port | Detected by checking that two port-type nodes are positioned on opposite faces (equal distance to opposite walls in every dimension), OR declared explicitly in the file |
| **junction** | An internal node where multiple struts meet | Connected to **more than 1** other node (degree > 1) |

A `point_status` ('active' vs 'suppressed') is also assigned: any node not referenced in any edge is marked 'suppressed'.

---

### Inputs

| Parameter | Type | Description |
|---|---|---|
| `filename` | `str` | Path to the `.dat` file |
| `types_and_status` | `bool` | If `False`, return only raw geometry. If `True` (default), also classify every node and return its type and status. |

### Outputs (with `types_and_status=True`)

| Return value | Type | Description |
|---|---|---|
| `points` | `np.ndarray` shape `(N, 3)` | 3-D coordinates, reordered so ports come first, then counterports, then junctions |
| `connectivity` | `list of [int, int]` | Edges as pairs of indices into `points` (re-mapped to the new order) |
| `point_types` | `list of str` | One of `'port'`, `'counterport'`, `'junction'` for every node |
| `point_status` | `list of str` | `'active'` or `'suppressed'` for every node |

In [ ]:
# ── Paste the read_lattice_file function here so the notebook is self-contained ──

import os
os.chdir(r"c:\Users\rimaz\Desktop\Semester Project- Intertwined Metatmaterials\intertwined-lattices")

def read_lattice_file(filename, types_and_status=True):
    with open(filename, 'r') as file:
        lines = file.readlines()

    points = []
    connectivity = []
    reading_points = True
    reading_counterports = False
    counterports = []

    for line in lines:
        if line.strip() == 'connectivity':
            reading_points = False
            continue
        if line.strip() == 'counterports':
            reading_counterports = True
            continue
        if reading_points:
            points.append(list(map(float, line.strip().split(','))))
        elif reading_counterports:
            counterports.append(list(map(int, line.strip().split(','))))
        else:
            connectivity.append(list(map(int, line.strip().split(','))))

    points = np.array(points)

    if not types_and_status:
        return points, connectivity

    point_types = ['port'] * len(points)
    point_status = ['active'] * len(points)
    if reading_counterports:
        for counterport in counterports:
            point_types[counterport[1]] = 'counterport'
    connection_count = np.zeros(len(points), dtype=int)

    for start, end in connectivity:
        connection_count[start] += 1
        connection_count[end] += 1

    for i, count in enumerate(connection_count):
        if count > 1:
            point_types[i] = 'junction'

    min_coords = np.min(points, axis=0)
    max_coords = np.max(points, axis=0)

    def are_counterports(p, q):
        positive_delta = [np.abs(p[i] - q[i]) for i in range(3)]
        for i in range(3):
            if max_coords[i] - min_coords[i] < 1e-6:
                continue
            if positive_delta[i] < 1e-6:
                if p[i] < min_coords[i] + 1e-6 or p[i] > max_coords[i] - 1e-6:
                    return False
            else:
                if np.abs(positive_delta[i] - (max_coords[i] - min_coords[i])) > 1e-6:
                    return False
        return True

    P = point_types.count('port') + point_types.count('counterport')
    PP = P + 1 if P % 2 != 0 else P
    ports = []
    if not reading_counterports:
        counterports = [np.zeros(3) for _ in range(PP // 2)]
    junctions = []
    idxs = []

    for i, point in enumerate(points):
        if point_types[i] == 'junction':
            junctions.append(point)
            idxs.append(P + len(junctions) - 1)
        if point_types[i] == 'port':
            found = False
            if not reading_counterports:
                for j, port in enumerate(ports):
                    if are_counterports(port, point):
                        counterports[j] = point
                        idxs.append(PP // 2 + j)
                        found = True
                        break
            if not found:
                ports.append(point)
                idxs.append(len(ports) - 1)
        if point_types[i] == 'counterport':
            idxs.append(-1)

    if reading_counterports:
        reading = 0
        for i, point in enumerate(points):
            if point_types[i] == 'counterport':
                idxs[i] = (PP // 2 + idxs[counterports[reading][0]])
                reading += 1

    connectivity = [[idxs[start], idxs[end]] for start, end in connectivity]
    if reading_counterports:
        counterports = [points[i] for i in range(P + len(junctions)) if idxs[i] >= PP // 2 and idxs[i] < P]

    points = np.concatenate((ports, counterports, junctions))
    point_types = ['port'] * (PP // 2) + ['counterport'] * (P // 2) + ['junction'] * (len(points) - P)

    connected_points = set([idx for edge in connectivity for idx in edge])
    for i in range(len(point_types)):
        if i not in connected_points:
            point_status[i] = 'suppressed'

    return points, connectivity, point_types, point_status


# ── Call the function on the sample file ──
points, connectivity, point_types, point_status = read_lattice_file('lattice_test.dat')

print("=== points (3-D coordinates) ===")
for i, (p, t, s) in enumerate(zip(points, point_types, point_status)):
    print(f"  node {i:2d}: {p}  type={t:<12s}  status={s}")

print(f"\n=== connectivity (edges) ===")
for edge in connectivity:
    print(f"  {edge}")

print(f"\nTotal nodes    : {len(points)}")
print(f"Total edges    : {len(connectivity)}")
print(f"Ports          : {point_types.count('port')}")
print(f"Counterports   : {point_types.count('counterport')}")
print(f"Junctions      : {point_types.count('junction')}")
print(f"Suppressed     : {point_status.count('suppressed')}")

## 3 — Bounding Box Utilities: `compute_bounding_planes()` and `compute_translation_vectors()`

These two small helper functions describe the **physical size of the unit cell** — they are used by `reframe_lattice()` and `tessellate_space()`.

### `compute_bounding_planes(points, shift)`

Figures out the six walls (faces) of the axis-aligned box surrounding the point cloud.

**Inputs**
- `points`: `np.ndarray (N, 3)` — the 3-D node coordinates
- `shift`: optional `np.ndarray (3,)` — offsets the planes by a translation vector (used when computing the *shifted* box for `reframe_lattice`)

**Output**  
A list of `(normal_vector, d)` tuples — one per non-degenerate face. A point `x` lies on a plane when `dot(normal, x) == d`.
If the lattice is flat in one dimension (e.g. a 2-D lattice), that dimension's planes are omitted.

---

### `compute_translation_vectors(min_coords, max_coords)`

Returns the **side lengths** of the bounding box as vectors along each axis. These are the distances you need to shift a copy of the cell so it sits immediately adjacent — i.e., the lattice **repeat vectors**.

**Inputs**: `min_coords`, `max_coords` — both `np.ndarray (3,)` corners of the bounding box  
**Output**: list of `np.ndarray (3,)` — one vector per non-degenerate axis (always the same vector duplicated twice per axis, once per face)

In [ ]:
def compute_bounding_planes(points, shift=np.array([0.0, 0.0, 0.0])):
    min_coords = np.min(points, axis=0)
    max_coords = np.max(points, axis=0)
    planes = []
    if not np.isclose(max_coords[0] - min_coords[0], 0):
        planes.append((np.array([1, 0, 0]), min_coords[0] + shift[0]))
        planes.append((np.array([-1, 0, 0]), -max_coords[0] - shift[0]))
    if not np.isclose(max_coords[1] - min_coords[1], 0):
        planes.append((np.array([0, 1, 0]), min_coords[1] + shift[1]))
        planes.append((np.array([0, -1, 0]), -max_coords[1] - shift[1]))
    if not np.isclose(max_coords[2] - min_coords[2], 0):
        planes.append((np.array([0, 0, 1]), min_coords[2] + shift[2]))
        planes.append((np.array([0, 0, -1]), -max_coords[2] - shift[2]))
    return planes


def compute_translation_vectors(min_coords, max_coords):
    translation_vectors = [
        np.array([max_coords[0] - min_coords[0], 0, 0]),
        np.array([max_coords[0] - min_coords[0], 0, 0]),
        np.array([0, max_coords[1] - min_coords[1], 0]),
        np.array([0, max_coords[1] - min_coords[1], 0]),
        np.array([0, 0, max_coords[2] - min_coords[2]]),
        np.array([0, 0, max_coords[2] - min_coords[2]])
    ]
    return [vec for vec in translation_vectors if not np.allclose(vec, 0)]


# ── Demo ──
planes = compute_bounding_planes(points)
print("Bounding planes  (normal_vector, offset d):")
axis_labels = ['x-min', 'x-max', 'y-min', 'y-max', 'z-min', 'z-max']
for label, (n, d) in zip(axis_labels, planes):
    print(f"  {label}: normal={n}  d={d:.4f}   →  plane equation: {n}·x = {d:.4f}")

min_coords = np.min(points, axis=0)
max_coords = np.max(points, axis=0)
print(f"\nBounding box: min={min_coords},  max={max_coords}")

tvecs = compute_translation_vectors(min_coords, max_coords)
print(f"\nRepeat (translation) vectors — one copy of each axis:")
for v in tvecs[::2]:   # deduplicate (each axis appears twice)
    print(f"  {v}")

## 4 — Reframing the Lattice: `reframe_lattice()`

### What is "reframing"?

A unit cell definition is arbitrary — you can slide the bounding box window over the lattice to start and end at different positions.
When you do that, some edges that were fully inside the old box now **stick out** of the new box, and edges from the next copy of the cell **poke in**.

`reframe_lattice()` handles this by:
1. Computing the **shifted bounding box** (the new frame).
2. For each edge, checking whether both endpoints, one, or neither lies outside the new frame.
3. If an endpoint is outside, it either **translates** it back inside (wraps it around) or **splits** the edge at the box boundary and adds the translated fragment.

The result is the same lattice topology but expressed relative to a different box origin.

---

### Inputs

| Parameter | Type | Description |
|---|---|---|
| `points` | `np.ndarray (N, 3)` | Node coordinates |
| `connectivity` | `list of [int, int]` | Edges |
| `translation_vector` | `np.ndarray (3,)` | How far to shift the box origin |
| `tolerance` | `float` | Numerical tolerance for coincident-point detection (default `1e-8`) |

### Output

| Return value | Type | Description |
|---|---|---|
| `new_points` | `np.ndarray` | Coordinates in the new frame |
| `new_connectivity` | `list of [int, int]` | Edges referencing the new point indices |

In [ ]:
def reframe_lattice(points, connectivity, translation_vector, tolerance=1e-8):

    def process_connectivity(ps, conns, plane, translation):
        normal, d = plane
        new_conns = []
        for conn in conns:
            p1, p2 = ps[conn[0]], ps[conn[1]]
            if np.dot(normal, p1) < d:
                if np.dot(normal, p2) < d:
                    ps[conn[0]] = p1 + translation
                    ps[conn[1]] = p2 + translation
                    new_conns.append(conn)
                else:
                    direction = p2 - p1
                    denom = np.dot(normal, direction)
                    t = (d - np.dot(normal, p1)) / denom
                    intersection = p1 + t * direction
                    if np.allclose(intersection, p2, atol=tolerance):
                        ps[conn[0]] = None
                        ps[conn[1]] = None
                        ps.append(p1 + translation)
                        ps.append(intersection + translation)
                        new_conns.append([len(ps) - 2, len(ps) - 1])
                    else:
                        ps[conn[0]] = intersection
                        ps.append(p1 + translation)
                        ps.append(intersection + translation)
                        new_conns.append(conn)
                        new_conns.append([len(ps) - 2, len(ps) - 1])
            else:
                if np.dot(normal, p2) < d:
                    direction = p1 - p2
                    denom = np.dot(normal, direction)
                    t = (d - np.dot(normal, p2)) / denom
                    intersection = p2 + t * direction
                    if np.allclose(intersection, p1, atol=tolerance):
                        ps[conn[0]] = None
                        ps[conn[1]] = None
                        ps.append(p2 + translation)
                        ps.append(intersection + translation)
                        new_conns.append([len(ps) - 2, len(ps) - 1])
                    else:
                        ps[conn[1]] = intersection
                        ps.append(p2 + translation)
                        ps.append(intersection + translation)
                        new_conns.append(conn)
                        new_conns.append([len(ps) - 2, len(ps) - 1])
                else:
                    new_conns.append(conn)
        return ps, new_conns

    shifted_bounding_planes = compute_bounding_planes(points, shift=translation_vector)
    translations = compute_translation_vectors(np.min(points, axis=0), np.max(points, axis=0))

    new_points = []
    new_connectivity = []

    for conn in connectivity:
        temp_conn = [[0, 1]]
        temp_points = [points[conn[0]], points[conn[1]]]
        for plane, translation in zip(shifted_bounding_planes, translations):
            temp_points, temp_conn = process_connectivity(temp_points, temp_conn, plane, translation)

        indices = []
        for pt in temp_points:
            found = False
            if pt is None:
                found = True
                indices.append(-1)
                continue
            for i, npt in enumerate(new_points):
                if np.allclose(pt, npt, atol=tolerance):
                    indices.append(i)
                    found = True
                    break
            if not found:
                new_points.append(pt)
                indices.append(len(new_points) - 1)
        for conn_pair in temp_conn:
            idx0 = indices[conn_pair[0]]
            idx1 = indices[conn_pair[1]]
            if [idx0, idx1] not in new_connectivity and [idx1, idx0] not in new_connectivity:
                new_connectivity.append([idx0, idx1])

    return np.array(new_points), new_connectivity


# ── Demo: shift the frame by (0.25, 0.25, 0.25) ──
translation_vector = np.array([0.25, 0.25, 0.25])
new_points, new_connectivity = reframe_lattice(points, connectivity, translation_vector)

print("Original lattice:")
print(f"  {len(points)} nodes,  {len(connectivity)} edges")

print(f"\nReframed lattice (shift = {translation_vector}):")
print(f"  {len(new_points)} nodes,  {len(new_connectivity)} edges")
print("\nNew node coordinates:")
for i, p in enumerate(new_points):
    print(f"  node {i}: {np.round(p,4)}")

## 5 — Visualising the 3-D Lattice: `plot_lattice()`

This is a straightforward plotting helper that renders the lattice in 3-D.

### Colour code

| Colour | Marker | Node type |
|---|---|---|
| Red circles | `o` (small → growing) | **ports** |
| Blue circles | `o` (small → growing) | **counterports** |
| Green squares | `s` | **junctions** |
| Black circles | `o` | all nodes (when no type list is given) |

Labels (0-based counters per type) are placed next to each node.

---

### Inputs

| Parameter | Type | Description |
|---|---|---|
| `points` | `np.ndarray (N, 3)` | 3-D coordinates |
| `connectivity` | `list of [int, int]` | Edges |
| `point_types` | `list of str` or `False` | If provided, colours nodes by type; otherwise all nodes are black |

### Output

An interactive 3-D `matplotlib` figure — no return value.

In [ ]:
def plot_lattice(points, connectivity, point_types=False):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')

    s_p = 50; s_cp = 50
    j_num = 0; p_num = 0; cp_num = 0
    shift = 0.1

    if point_types:
        for point, point_type in zip(points, point_types):
            if point_type == 'port':
                ax.scatter(point[0], point[1], point[2], c='r', marker='o', s=s_p)
                ax.text(point[0]+shift, point[1]+shift, point[2], str(p_num), color='black')
                p_num += 1; s_p *= 2
            elif point_type == 'counterport':
                ax.scatter(point[0], point[1], point[2], c='b', marker='o', s=s_cp)
                ax.text(point[0]+shift, point[1]+shift, point[2], str(cp_num), color='black')
                cp_num += 1; s_cp *= 2
            elif point_type == 'junction':
                ax.scatter(point[0], point[1], point[2], c='g', marker='s', s=200)
                ax.text(point[0]+shift, point[1]+shift, point[2], str(j_num), color='black')
                j_num += 1
    else:
        ax.scatter(points[:, 0], points[:, 1], points[:, 2], c='k', marker='o')

    lines = [(points[start], points[end]) for start, end in connectivity]
    line_collection = Line3DCollection(lines, colors='k', linewidths=2)
    ax.add_collection3d(line_collection)

    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_box_aspect([1, 1, 1])
    plt.tight_layout()
    plt.show()


# ── Plot with type colouring ──
print("Red = ports  |  Blue = counterports  |  Green squares = junctions")
plot_lattice(points, connectivity, point_types)

## 6 — Finding Symmetries: `find_symmetries()` and `find_symmetries_v1()`

### What is a symmetry here?

A **symmetry** of the point cloud is a way of **renaming** (permuting) the nodes such that every pairwise distance is exactly preserved.
Geometrically this corresponds to rotations, reflections, and combinations of the two.
The result is expressed as a **permutation list**: index `i` maps to `perm[i]`.

For example `[1, 0, 2, 3]` means "node 0 maps to node 1, node 1 maps to node 0, nodes 2 and 3 stay."

---

### How it works (the graph-isomorphism trick)

Both functions encode the point distances as a **complete weighted graph** — every pair of nodes gets an edge whose weight equals their Euclidean distance.
A distance-preserving permutation is then exactly a **graph automorphism** (a mapping of the graph to itself that preserves edge weights).
NetworkX's `GraphMatcher` finds all of them.

---

### `find_symmetries(points, point_status=None)`

Uses **only pairwise distances** — no connectivity required.  
Suppressed nodes (not part of any edge) get weight-0 edges so they are effectively isolated from the rest.

**Returns**: list of permutation lists — every distance-preserving relabelling of the nodes.

---

### `find_symmetries_v1(points, connectivity, point_status=None)`

Stricter version: first finds all distance-preserving permutations, then **filters** to keep only those that also preserve the **actual edge set** of the lattice.

**Returns**: same format, but only symmetries that also keep the connectivity graph intact.

In [ ]:
def find_symmetries(points, point_status=None):
    if point_status is None:
        point_status = ['active'] * len(points)

    G = nx.Graph()
    G.add_nodes_from(range(len(points)))
    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            if point_status[i] == 'suppressed' or point_status[j] == 'suppressed':
                G.add_edge(i, j, weight=0)
            else:
                G.add_edge(i, j, weight=np.linalg.norm(points[i] - points[j]))

    def edge_match(e1, e2):
        return np.isclose(e1['weight'], e2['weight'])

    GM = GraphMatcher(G, G, edge_match=edge_match)
    valid_permutations = list(GM.isomorphisms_iter())

    formatted_permutations = []
    for perm in valid_permutations:
        formatted_perm = [0] * len(points)
        for key, value in perm.items():
            formatted_perm[key] = value
        formatted_permutations.append(formatted_perm)
    return formatted_permutations


def find_symmetries_v1(points, connectivity, point_status=None):
    if point_status is None:
        point_status = ['active'] * len(points)

    G = nx.Graph()
    G.add_nodes_from(range(len(points)))
    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            if point_status[i] == 'suppressed' or point_status[j] == 'suppressed':
                G.add_edge(i, j, weight=0)
            else:
                G.add_edge(i, j, weight=np.linalg.norm(points[i] - points[j]))

    def edge_match(e1, e2):
        return np.isclose(e1['weight'], e2['weight'])

    GM = GraphMatcher(G, G, edge_match=edge_match)
    valid_permutations = list(GM.isomorphisms_iter())

    edges_set = set(tuple(sorted(edge)) for edge in connectivity)
    valid_permutations = [
        perm for perm in valid_permutations
        if set(tuple(sorted((perm[start], perm[end]))) for start, end in connectivity) == edges_set
    ]

    formatted_permutations = []
    for perm in valid_permutations:
        formatted_perm = [0] * len(points)
        for key, value in perm.items():
            formatted_perm[key] = value
        formatted_permutations.append(formatted_perm)
    return formatted_permutations


# ── Demo ──
symm = find_symmetries(points, point_status)
symm_v1 = find_symmetries_v1(points, connectivity, point_status)

print(f"find_symmetries()    → {len(symm)} symmetries (distance-only)")
print(f"find_symmetries_v1() → {len(symm_v1)} symmetries (distance + connectivity)")

print("\nFirst 5 permutations from find_symmetries():")
for p in symm[:5]:
    print(f"  {p}")

print("\nFirst 5 permutations from find_symmetries_v1():")
for p in symm_v1[:5]:
    print(f"  {p}")

## 7 — Port & Counterport Mapping: `generate_counterports()`

### Concept

Ports and counterports are **periodic twins** — they are the nodes through which the unit cell connects to its copies when the lattice is tiled.
Each port sits on one face of the bounding box; its counterport sits on the opposite face at the same relative position.
Together, they define the **repeat vectors** of the lattice.

This standalone helper simply builds a bi-directional lookup dictionary from a list of node types.

---

### Input

| Parameter | Type | Description |
|---|---|---|
| `types` | `list of str` | One entry per node: `'port'`, `'counterport'`, or `'junction'` |

### Output

A `dict` mapping each port or counterport **index** to its paired counterpart's index.

```
{
  port_0_index   : counterport_0_index,
  counterport_0_index : port_0_index,
  port_1_index   : counterport_1_index,
  ...
}
```

The pairing is index-offset based: if there are P ports (indices 0 … P-1), their counterparts live at indices P … 2P-1.

In [ ]:
def generate_counterports(types):
    counterports = {}
    P = sum([1 for t in types if t == 'port'])
    for i, t in enumerate(types):
        if t == 'port':
            counterports[i] = i + P
        elif t == 'counterport':
            counterports[i] = i - P
    return counterports


# ── Demo ──
cp_map = generate_counterports(point_types)

print("Port / counterport pairing dictionary:")
print("  {node_index : paired_node_index}")
print()
for k, v in cp_map.items():
    type_k = point_types[k]
    type_v = point_types[v]
    print(f"  node {k} ({type_k:<12s})  ↔  node {v} ({type_v})")

## 8 — Space Tessellation: `tessellate_space()`

### Concept

A lattice unit cell by itself is just one repeating tile.
To simulate a real material you need to **tile** many copies of it side by side.
`tessellate_space()` does this by translating the entire unit cell along its repeat vectors (the port→counterport displacement vectors) and stacking the copies together.

The integer `N` controls how many copies to place in each periodic direction.
With 3 independent directions and `N=2` you get $2^3 = 8$ copies; with `N=3` you get $3^3 = 27$.

### Step-by-step logic

1. Determine the **repeat vectors** from the port → counterport displacements.
2. Generate every combination of integer multiples $(m_1, m_2, m_3)$ with $m_i \in \{0, 1, \ldots, N-1\}$ using `itertools.product`.
3. For each combination compute the total translation $v = m_1 v_1 + m_2 v_2 + m_3 v_3$.
4. Shift all points by $v$, add new points (avoiding duplicates), and add translated copies of every edge.

---

### Inputs

| Parameter | Type | Description |
|---|---|---|
| `points` | `np.ndarray` | Unit-cell node coordinates |
| `connectivities` | `list of [int, int]` | Unit-cell edges |
| `point_types` | `list of str` | Node types (used to identify ports and counterports) |
| `N` | `int` | Number of copies per periodic direction |

### Outputs

| Return | Type | Description |
|---|---|---|
| `tessellated_points` | `np.ndarray` | All node coordinates of the tiled lattice |
| `tessellated_connectivities` | `list of [int, int]` | All edges of the tiled lattice |

In [ ]:
def tessellate_space(points, connectivities, point_types, N):
    tessellated_points = points.tolist()
    tessellated_connectivities = []
    vectors = []
    counterports = generate_counterports(point_types)

    for ip, (p, t) in enumerate(zip(points, point_types)):
        if t == 'port':
            icp = counterports[ip]
            cp = points[icp]
            vectors.append(cp - p)

    multiplicities = list(product(range(N), repeat=len(vectors)))
    point_index_map = {tuple(p): i for i, p in enumerate(tessellated_points)}

    for mult in multiplicities:
        v = np.zeros(3)
        for i, m in enumerate(mult):
            v += m * vectors[i]
        translated_points = points + v

        for tp in translated_points:
            tp_tuple = tuple(tp)
            if tp_tuple not in point_index_map:
                point_index_map[tp_tuple] = len(tessellated_points)
                tessellated_points.append(tp.tolist())

        tessellated_connectivities.extend(
            [[point_index_map[tuple(points[start] + v)], point_index_map[tuple(points[end] + v)]]
             for start, end in connectivities]
        )

    return np.array(tessellated_points), tessellated_connectivities


# ── Demo: tile 2×2×2 ──
N = 2
tess_points, tess_conn = tessellate_space(points, connectivity, point_types, N)

print(f"Unit cell   : {len(points)} nodes,  {len(connectivity)} edges")
print(f"Tiled N={N}  : {len(tess_points)} nodes,  {len(tess_conn)} edges")

print(f"\nVisualising the N={N} tessellation...")
plot_lattice(tess_points, tess_conn)

## 9 — The `Graph` Class: Structure and Attributes

The `Graph` class is the **central data container** for the whole project.
When you instantiate it, it immediately:

1. Stores the raw inputs (`points`, `connections`, `types`, `status`)
2. Separates node indices into `ports` (includes counterports) and `junctions`
3. Calls `generate_edges()`, `generate_counterports()`, `generate_counteredges()` to pre-compute all relational data
4. Calls `find_symmetries()` so the symmetry group is ready to use

### Key attributes after construction

| Attribute | Type | What it holds |
|---|---|---|
| `points` | `np.ndarray` | 3-D coordinates of all nodes |
| `connections` | `list` | Raw edge list as read from the file |
| `types` | `list of str` | `'port'`, `'counterport'`, or `'junction'` per node |
| `status` | `list of str` | `'active'` or `'suppressed'` per node |
| `ports` | `list of int` | Indices of all port/counterport nodes |
| `junctions` | `list of int` | Indices of all junction nodes |
| `edges` | `dict {id: (n1,n2)}` | Numbered edge dictionary |
| `counterports` | `dict {i: j}` | Port ↔ counterport index pairs |
| `counteredges` | `dict {id: id}` | Maps each edge to its periodic twin |
| `translations` | `dict {id: vec}` | Translation vector separating an edge from its twin |
| `symmetries` | `list of lists` | All distance-preserving node permutations |

In [ ]:
class Graph:
    def __init__(self, points, connections, types, status):
        self.points = points
        self.connections = connections
        self.types = types
        self.status = status
        self.ports = [i for i, t in enumerate(types) if t == 'port' or t == 'counterport']
        self.junctions = [i for i, t in enumerate(types) if t == 'junction']
        self.edges = self.generate_edges()
        self.counterports = self.generate_counterports()
        self.translations = {}
        self.counteredges = self.generate_counteredges()
        self.symmetries = find_symmetries(points, status)

    def generate_edges(self):
        edges = defaultdict(int)
        for i, (n1, n2) in enumerate(self.connections):
            edges[i] = tuple(sorted((n1, n2)))
        return edges

    def generate_counterports(self):
        counterports = {}
        P = sum([1 for t in self.types if t == 'port'])
        for i, t in enumerate(self.types):
            if t == 'port':
                counterports[i] = i + P
            elif t == 'counterport':
                counterports[i] = i - P
        return counterports

    def generate_counteredges(self):
        edge2id = {v: k for k, v in self.edges.items()}
        counteredges = {}
        translations = {}
        for edge_id, (n1, n2) in self.edges.items():
            if n1 in self.ports:
                if tuple(sorted((self.counterports[n1], n2))) in edge2id:
                    counteredges[edge_id] = edge2id[tuple(sorted((self.counterports[n1], n2)))]
                    translations[edge_id] = self.points[n1] - self.points[self.counterports[n1]]
                else:
                    counteredges[edge_id] = edge_id
                    translations[edge_id] = np.zeros(len(self.points[0]))
            elif n2 in self.ports:
                if tuple(sorted((n1, self.counterports[n2]))) in edge2id:
                    counteredges[edge_id] = edge2id[tuple(sorted((n1, self.counterports[n2])))]
                    translations[edge_id] = self.points[n2] - self.points[self.counterports[n2]]
                else:
                    counteredges[edge_id] = edge_id
                    translations[edge_id] = np.zeros(len(self.points[0]))
            else:
                counteredges[edge_id] = edge_id
                translations[edge_id] = np.zeros(len(self.points[0]))
        self.counteredges = counteredges
        self.translations = translations
        return counteredges

    def find_minimum_internal_nodes(self, edges, start_node, end_node):
        graph = defaultdict(list)
        for edge_id, (node1, node2) in edges.items():
            graph[node1].append(node2)
            graph[node2].append(node1)
        queue = [(start_node, [start_node])]
        visited = set()
        while queue:
            current_node, path = queue.pop(0)
            if current_node in visited:
                continue
            visited.add(current_node)
            if current_node == end_node:
                return path[1:-1]
            for neighbor in graph[current_node]:
                if neighbor not in visited:
                    queue.append((neighbor, path + [neighbor]))
        return []

    def junction2ports(self):
        junction_to_ports = defaultdict(list)
        for edge_id, (n1, n2) in self.edges.items():
            if n1 in self.junctions:
                if n2 in self.ports:
                    junction_to_ports[n1].append(n2)
            if n2 in self.junctions:
                if n1 in self.ports:
                    junction_to_ports[n2].append(n1)
        return dict(junction_to_ports)

    def plot_graph(self, N):
        G = nx.Graph()
        for node in self.points:
            G.add_node(tuple(node) if isinstance(node, np.ndarray) else node)
        edge_labels = {}
        for edge_id, (n1, n2) in self.edges.items():
            G.add_edge(n1, n2)
            edge_labels[(n1, n2)] = f"{N}"
        pos = nx.spring_layout(G)
        nx.draw(G, pos, with_labels=True, node_color='grey', node_size=500, font_size=10, font_weight='bold')
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red', font_size=8)
        plt.title("Graph Visualization")
        plt.show()


# ── Instantiate the Graph ──
g = Graph(points, connectivity, point_types, point_status)

print("Graph attributes after construction:")
print(f"  Number of nodes      : {len(g.points)}")
print(f"  Port/counterport idx : {g.ports}")
print(f"  Junction idx         : {g.junctions}")
print(f"\n  edges dict           : {dict(g.edges)}")
print(f"\n  counterports dict    : {g.counterports}")
print(f"\n  counteredges dict    : {g.counteredges}")
print(f"\n  translations dict    :")
for eid, vec in g.translations.items():
    print(f"    edge {eid}: {vec}")
print(f"\n  Number of symmetries : {len(g.symmetries)}")

## 10 — Graph Methods: Edges, Counterports, Counteredges, and More

### `generate_edges()`

Converts the raw `connections` list into a numbered dictionary:

```
{edge_id: (smaller_node_index, larger_node_index), ...}
```

Sorting the pair makes edge lookup order-independent (edge (2,5) == edge (5,2)).

---

### `generate_counterports()`

Identical to the standalone `generate_counterports()` function — builds the port ↔ counterport index mapping for this particular `Graph` instance.

---

### `generate_counteredges()`

For each edge in the graph, finds the **periodic twin** — the equivalent edge that would appear in the adjacent copy of the unit cell.

If an edge connects a port (or counterport) to a junction, the twin is the edge connecting the *other* boundary node (on the opposite face) to the same junction.
Also records the **translation vector** separating the two periodic twins.

**Output**: stored in `self.counteredges` (dict `{edge_id: twin_edge_id}`) and `self.translations` (dict `{edge_id: np.ndarray}`).

---

### `find_minimum_internal_nodes(edges, start_node, end_node)`

Runs a **Breadth-First Search (BFS)** from `start_node` to `end_node` through the edge graph and returns the list of intermediate nodes along the shortest path (excluding the start and end themselves).

Used internally by `plot_lattice_graph()` to decide how to position counterport nodes in the 2-D schematic.

---

### `junction2ports()`

Returns a dictionary mapping each junction to the list of ports it is directly connected to.
Used for computing the 2-D layout of the graph diagram.

In [ ]:
# ── generate_edges() output ──
print("=== generate_edges() ===")
print("  edge_id : (node_a, node_b)")
for eid, pair in g.edges.items():
    print(f"   {eid:2d}  :  {pair}")

# ── generate_counterports() output ──
print("\n=== generate_counterports() ===")
print("  port_index  →  counterport_index")
for k, v in g.counterports.items():
    print(f"   node {k} ({point_types[k]:<12s})  →  node {v} ({point_types[v]})")

# ── generate_counteredges() output ──
print("\n=== generate_counteredges() ===")
print("  edge_id  →  twin_edge_id   |  translation vector")
for eid, twin in g.counteredges.items():
    print(f"   {eid:2d}  →  {twin:2d}   |  {g.translations[eid]}")

# ── junction2ports() ──
print("\n=== junction2ports() ===")
j2p = g.junction2ports()
for junction, connected_ports in j2p.items():
    print(f"   junction {junction}  →  ports {connected_ports}")

# ── find_minimum_internal_nodes() ──
print("\n=== find_minimum_internal_nodes() ===")
# Find shortest path between node 0 (port) and its counterport
port_0 = g.ports[0]
counterport_0 = g.counterports[port_0]
path = g.find_minimum_internal_nodes(g.edges, port_0, counterport_0)
print(f"   Intermediate nodes between node {port_0} and node {counterport_0}: {path}")

## 11 — Graph Methods: Visualisation

The `Graph` class provides two different ways to visualise the lattice as an **abstract graph** (not the 3-D geometry, but the connectivity diagram).

---

### `plot_lattice_graph()`

A styled 2-D schematic where:
- **Junction nodes** are placed at the centre (or on a circle if there are multiple).
- **Port nodes** are placed around the circumference, spread evenly.
- **Counterport nodes** are placed at the mirror positions (opposite centre or mirrored across the junction axis).
- Node labels use LaTeX-formatted subscripts ($p_1$, $j_1$, etc.).
- Suppressed nodes are drawn in light grey.

This gives a clean, publication-style diagram of the abstract connectivity of the unit cell.

---

### `plot_graph(N)`

A simpler fallback using NetworkX's `spring_layout` — nodes are placed to minimise edge crossing, with edge labels showing the number `N`.
Useful for quick inspection when the geometric placement of `plot_lattice_graph()` is not critical.

In [ ]:
# ── plot_graph(N) — simple NetworkX spring-layout view ──
print("plot_graph(N=1): NetworkX spring-layout of the connectivity")
g.plot_graph(N=1)

## 12 — Main Execution Flow (`if __name__ == '__main__':`)

When you run `symmetry.py` directly (not imported), it executes a demo workflow. The active (non-commented) steps are:

1. **Read** `lattice_test.dat` with `types_and_status=False` — gets raw geometry only.
2. **Plot** the raw 3-D lattice.
3. **Find symmetries** of the point cloud (no connectivity constraint).
4. **Print** the permutations and their count.
5. Call `sys.exit()` — everything below that line (tessellation, noise injection, file export) is currently disabled.

The cells below reproduce this exact workflow step by step.

In [ ]:
# Step 1 — Read raw geometry only (types_and_status=False)
# Returns only points and connectivity, no type/status classification
point_raw, conn_raw = read_lattice_file('lattice_test.dat', types_and_status=False)

print("Raw points (no type classification):")
for i, p in enumerate(point_raw):
    print(f"  node {i}: {p}")

print(f"\nRaw connectivity ({len(conn_raw)} edges):")
for e in conn_raw:
    print(f"  {e}")

In [ ]:
# Step 2 — Plot the raw 3-D lattice (no type colouring)
# All nodes appear as black dots; all edges as black lines
print("Plotting raw lattice (no type colouring)...")
plot_lattice(point_raw, conn_raw)

In [ ]:
# Step 3 — Find all distance-preserving symmetries of the raw point cloud
# (no connectivity constraint — same as the active __main__ call)
symm_raw = find_symmetries(point_raw)

print(f"Number of symmetries found: {len(symm_raw)}")
print("\nFull list of symmetry permutations")
print("Each row is a permutation: perm[i] = j means node i maps to node j")
print()
for idx, perm in enumerate(symm_raw):
    print(f"  sym {idx:3d}: {perm}")

## Summary — How It All Fits Together

```
lattice_test.dat
        │
        ▼
read_lattice_file()
   ├── returns: points, connectivity, point_types, point_status
   └── classifies each node as port / counterport / junction
        │
        ├──► plot_lattice()                 ← visualise the 3-D unit cell
        │
        ├──► find_symmetries()              ← find all distance-preserving node permutations
        │         └── uses NetworkX GraphMatcher on a complete distance-weighted graph
        │
        ├──► tessellate_space()             ← tile N copies along each periodic direction
        │         └── uses generate_counterports() to get the repeat vectors
        │
        ├──► reframe_lattice()              ← shift the unit cell boundary to a new origin
        │         └── uses compute_bounding_planes() + compute_translation_vectors()
        │
        └──► Graph(points, conn, types, status)
                  ├── generate_edges()         → numbered edge dict
                  ├── generate_counterports()  → port ↔ counterport index map
                  ├── generate_counteredges()  → each edge ↔ its periodic twin + offset vector
                  ├── find_symmetries()        → symmetry group stored in g.symmetries
                  ├── junction2ports()         → which ports connect to each junction
                  ├── find_minimum_internal_nodes()  → BFS path helpers
                  ├── plot_graph()             → quick NetworkX layout diagram
                  └── plot_lattice_graph()     → styled 2-D schematic with LaTeX labels
```